In [ ]:
import torch

!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster  --y
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git
!pip install torch-geometric-temporal

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 154.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 87.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 73.7 MB/s eta 0:00:00
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-hf4rwp5g
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-hf4rwp5g
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit a5b69c37a05561ebb92931b3d586d664a7269585
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for torch-geometric: filename=torch_geometric-2.8.0-py3-none-any.whl size

In [ ]:
import os
import pandas as pd
from functools import reduce
from copy import deepcopy as dc
import gc
import torch
import torch.nn as nn
import torch_geometric_temporal as torch_geom
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [ ]:
class WaitTimeDataset(Dataset):
  def __init__(self, X, y, seq_len=60, pred_len=15):
    self.X = torch.from_numpy(X)
    self.y = torch.from_numpy(y)
    self.seq_len = seq_len
    self.pred_len = pred_len

  def __len__(self):
    return len(self.X) - self.seq_len - self.pred_len

  def __getitem__(self, idx):
    return self.X[idx:idx + self.seq_len].float(), self.y[idx:idx + self.seq_len + self.pred_len - 1].float()

class WaitTimeSTGNN(nn.Module):
  def __init__(self, in_channels, out_channels, hidden_channels=64, K=2):
        super().__init__()
        self.gclstm1 = torch_geom.GConvLSTM(in_channels, hidden_channels, K)
        self.fc = nn.Linear(hidden_channels, out_channels)

  def forward(self, x, edge_index):
    batch_size, seq_len, num_nodes, num_features = x.shape

    h, c = None, None

    for t in range(seq_len):
      time_step = x[:, t, :, :].flatten(-1, num_features)
      h, c = self.gclstm1(time_step, edge_index, H=h, C=c)
      h = torch.relu(h)

    h = h.view(batch_size, num_nodes, -1)

    x = self.fc(h)
    return x.squeeze(-1)

In [ ]:
def load_data(folder_paths):
  dfs = []
  rides = []
  for folder_path in folder_paths:
    for file_name in os.listdir(f"{folder_path}Wait_Time_Features/"):
      ride_name = file_name.replace("_features.parquet", "")

      df = pd.read_parquet(f"{folder_path}Wait_Time_Features/{file_name}")
      df.set_index(pd.to_datetime(df["datetime_idx"].astype(str)), inplace=True)

      ride_cols = [
          "wait_time", "minutes_from_open", "minutes_to_close",
          "wait_time_lag_day", "wait_time_lag_week", "wait_time_lag_year",
          "wt_rolling_week_avg", "wt_rolling_month_avg", "wt_rolling_year_avg",
          "wt_rolling_week_max", "wt_rolling_month_max", "wt_rolling_year_max",
          "wt_rolling_week_std", "wt_rolling_month_std", "wt_rolling_year_std",
          "duration", "min_height", "has_express", "has_ea",
          "is_kids_ride", "is_thrill_ride", "is_dark_ride", "is_water_ride", "has_indoor_queue",
        ]

      universal_cols = [
          "temp", "rhum", "prcp", "wspd", "pres", "cldc", "coco",
          "is_holiday", "is_major_holiday", "is_annual_event",
          "year", "month_sin", "month_cos", "day_sin", "day_cos",
          "time_sin", "time_cos", "day_of_week_sin", "day_of_week_cos"
        ]

      df = df[ride_cols + universal_cols].astype(np.float16)

      df.columns = [col + f"_{ride_name}" for col in ride_cols] + universal_cols

      dfs.append(df)
      rides.append(ride_name)

  full_df = reduce(lambda left, right: pd.merge(left, right[right.columns.difference(left.columns)], left_index=True, right_index=True, how="outer"), dfs)
  del dfs
  gc.collect()

  rides_df = full_df.drop(columns=universal_cols).fillna(0)
  features_df = full_df[universal_cols].fillna(0)

  return rides_df, rides, ride_cols, features_df

def calculate_edge_index(target):
  corr_matrix = np.nan_to_num(np.corrcoef(target.T))
  adj_indices = np.where(corr_matrix > 0.7)

  edge_index = torch.tensor(np.array(adj_indices), dtype=torch.long)

  mask = edge_index[0] != edge_index[1]
  edge_index = edge_index[:, mask]

  return edge_index

def process_data(rides_df, num_rides, num_ride_cols, features_df):
  target_cols = []
  for col in rides_df.columns:
    if col.startswith("wait_time") and not col.startswith("wait_time_lag"):
      target_cols.append(col)

  scaler = MinMaxScaler(feature_range=(0, 1))
  ride_features = scaler.fit_transform(np.array(rides_df.drop(columns=target_cols)))
  universal_features = scaler.fit_transform(np.array(features_df))

  target = np.log1p(np.array(rides_df[target_cols]))
  ride_features = ride_features.reshape((-1, num_rides, num_ride_cols))
  universal_features = np.tile(universal_features[:, np.newaxis, :], (1, 37, 1))

  stgnn_features = np.concatenate([ride_features, universal_features], axis=-1)

  edge_index = calculate_edge_index(target)

  return target, stgnn_features, edge_index

def train_one_epoch(model, loader, optimizer, criterion, device, edge_index):
  model.train()
  edge_index = edge_index.to(device)
  for x, y in loader:
    x, y = x.to(device).float(), y.to(device).float()
    optimizer.zero_grad()
    logits = model(x, edge_index)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()

@torch.no_grad()
def evaluate(model, loader, device, edge_index):
  model.eval()
  edge_index = edge_index.to(device)
  preds = []
  tests = []
  for x, y in loader:
    x, y = x.to(device), y.numpy()
    pred = model(x, edge_index)
    preds.append(pred)
    tests.append(y)

  preds = np.concatenate(preds, axis=0).flatten()
  tests = np.concatenate(tests, axis=0).flatten()

  mae = mean_absolute_error(tests, pred)
  rmse = root_mean_squared_error(tests, pred)
  wape = sum(np.abs(tests - preds)) / sum(tests)

  return mae, rmse, wape

def train_model(train_loader, edge_index, num_epochs=50, lr=1e-3, wd=1e-3):
  device = torch.device("cpu")#('cuda' if torch.cuda.is_available() else 'cpu')
  print(f"Using: {device}")
  model = WaitTimeSTGNN(in_channels=42, out_channels=1, hidden_channels=64).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
  criterion = nn.MSELoss()
  for epoch in range(1, num_epochs + 1):
    train_one_epoch(model, train_loader, optimizer, criterion, device, edge_index)

  return model, device

In [ ]:
rides_df, rides, ride_cols, features_df = load_data(["/content/drive/MyDrive/Theme_Park_Data/Universal_IOA/", "/content/drive/MyDrive/Theme_Park_Data/Universal_Studios_FL/"])

KeyboardInterrupt: 

In [ ]:
features_df = pd.read_parquet("/content/drive/MyDrive/Theme_Park_Data/CNN_Features.parquet").astype(np.float16)
rides_df = pd.read_parquet("/content/drive/MyDrive/Theme_Park_Data/CNN_Ride_Data.parquet").astype(np.float16)

rides_df_train = rides_df[:len(rides_df) - 1020*365]
rides_df_test = rides_df[len(rides_df) - 1020*365:]

features_df_train = features_df[:len(rides_df) - 1020*365]
features_df_test = features_df[len(rides_df) - 1020*365:]

target_train, features_train, edge_index_train = process_data(rides_df_train, 37, 23, features_df_train)
target_test, features_test, edge_index_test = process_data(rides_df_test, 37, 23, features_df_test)

train_dataset = WaitTimeDataset(features_train, target_train)
test_dataset = WaitTimeDataset(features_test, target_test)

batch_size = 1
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
model, device = train_model(train_loader, edge_index_train)

Using: cpu


IndexError: Dimension out of range (expected to be in range of [-3, 2], but got 42)